# Crime Pattern Detection using Data Mining

Final Cleaned Notebook

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, silhouette_score

from mlxtend.frequent_patterns import fpgrowth, association_rules

sns.set(style="whitegrid")


In [ ]:

df = pd.read_csv("01_District_wise_crimes_committed_IPC_2001_2012.csv")
df.head()


In [ ]:

df.columns = df.columns.str.strip().str.lower()

df = df.rename(columns={
    'state/ut': 'state',
    'total ipc crimes': 'total_crimes',
    'hurt/grevious hurt': 'hurt'
})

df['state'] = df['state'].str.title()
df['district'] = df['district'].str.replace('ZZ TOTAL','TOTAL').str.title()
df.columns.tolist()


In [ ]:

df['crime_level'] = pd.qcut(
    df['total_crimes'],
    3,
    labels=['Low','Medium','High']
)
df[['total_crimes','crime_level']].head()


In [ ]:

FEATURES = [
    'murder','rape','theft',
    'riots','robbery','burglary',
    'kidnapping & abduction'
]

X = df[FEATURES]
y = df['crime_level']
X.shape, y.shape


In [ ]:

yearly = df.groupby('year')['total_crimes'].sum()
plt.plot(yearly, marker='o')
plt.title("Year-wise Crime Trend")
plt.show()

df[FEATURES].sum().plot(kind='pie', autopct='%1.1f%%')
plt.show()

top_states = df.groupby("state")['total_crimes'].sum().nlargest(10)
sns.barplot(x=top_states.values, y=top_states.index)
plt.show()

sns.heatmap(df[FEATURES].corr(), annot=True, cmap='coolwarm')
plt.show()


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_scaled, y)


In [ ]:

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df['cluster'] = kmeans.labels_
df.groupby('cluster')[FEATURES].mean().round(1)


In [ ]:

binary_df = df[FEATURES].apply(lambda x: x > x.median())

frequent_itemsets = fpgrowth(binary_df, min_support=0.3, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets[frequent_itemsets['length'] <= 3]

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.7
).sort_values(by='lift', ascending=False)

rules[['antecedents','consequents','support','confidence','lift']].head(10)


In [ ]:

y_pred = rf_model.predict(X_scaled)

print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))

cm = confusion_matrix(y, y_pred, labels=['Low','Medium','High'])
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Low','Medium','High'],
            yticklabels=['Low','Medium','High'])
plt.show()

sil = silhouette_score(X_scaled, kmeans.labels_)
print("Silhouette Score:", sil)


In [ ]:

os.makedirs("models", exist_ok=True)

with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("models/rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("models/kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans, f)

print("Models saved successfully")
